# Feature engineering & model preparation — Step 8

**Analysis-ready dataset for wage regression.**
Reads from `data/processed/feature_engineered/`, writes to `data/processed/feature_engineered/`. Does not modify any file produced by `data_cleaning.ipynb`.
Run from `Code/DataCleaning/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('.').resolve()  # Code/DataCleaning
# If Player_Data_Ready not found, try PROJECT_ROOT = BASE.parent.parent
PROJECT_ROOT = BASE.parent.parent
DATABASES = PROJECT_ROOT / 'data'
ANALYSIS_READY_DIR = PROJECT_ROOT / 'data' / 'processed' / 'feature_engineered'
MODELS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'feature_engineered'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_READY_FILES = {
    'Defenders': 'defenders.xlsx',
    'Midfielders_Forwards': 'midfielders_forwards.xlsx',
    'Goalkeepers': 'goalkeepers.xlsx',
}

## Step A — Load and apply sample filters

Player_Data_Ready files already have Salary_Matched=True only. Apply: **Min ≥ 90** (degenerate per-90 below this) and exclude **2 Teams** / **3 Teams** (multi-team aggregate rows).

In [ ]:
raw_dfs = {}
for name, fname in ANALYSIS_READY_FILES.items():
    path = ANALYSIS_READY_DIR / fname
    if not path.exists():
        print(f'NOT FOUND: {path}')
        continue
    df = pd.read_excel(path, engine='openpyxl')
    n0 = len(df)
    # Filter 1: Min >= 90 (degenerate per-90 values below this — see methodology)
    df['Min'] = pd.to_numeric(df['Min'], errors='coerce')
    df = df[df['Min'] >= 90]
    n1 = len(df)
    # Filter 2: Exclude multi-team aggregate rows (no valid single Team for fixed effects)
    df = df[~df['Team'].astype(str).str.strip().isin(['2 Teams', '3 Teams'])]
    n2 = len(df)
    print(f'{name}: {n0} -> {n1} (Min>=90) -> {n2} (excl. multi-team)')
    raw_dfs[name] = df.copy()

## Step B — Drop exact duplicate columns

Keep the left name, drop the right (100% identical). Columns not present are skipped.

In [ ]:
EXACT_DUPES = [
    'Sh_Standard',   # == Sh
    'Int_Def',      # == Int
    'CrdY_Detail',   # == CrdY
    'OG_Detail',     # == OG
    'PPM_Team',      # == PPM
    'Mn/MP_Detail',  # == Mn/MP
    'Off_Detail',    # == Off (Midfielders_Forwards only)
    'GA_Summary',    # == GA (Goalkeepers only)
    'PKvs_Summary', # == PKvs (Goalkeepers only)
]
META_COLS = [
    'Weekly Wages', 'Annual Wages', 'Salary_Notes', 'Rk',
]
for name, df in raw_dfs.items():
    to_drop = [c for c in EXACT_DUPES + META_COLS if c in df.columns]
    df = df.drop(columns=to_drop)
    raw_dfs[name] = df
    print(f'{name}: dropped {len(to_drop)} duplicate/meta cols -> {df.shape[1]} remain')

## Step C — Drop derived and collinear columns

Mathematically derived from other columns; including them causes multicollinearity.

In [ ]:
DERIVED_COLS = [
    'G+A', 'G-PK', 'Tkl+Int', '90s', 'Compl', 'L_Detail', 'Mn/MP', 'Mn/Start', 'Mn/Sub',
]
for name, df in raw_dfs.items():
    to_drop = [c for c in DERIVED_COLS if c in df.columns]
    df = df.drop(columns=to_drop)
    raw_dfs[name] = df
    print(f'{name}: dropped {len(to_drop)} derived cols -> {df.shape[1]} remain')

## Step D — Build per-90 features

Counting stats divided by 90s; then drop raw counts. Rate stats (G/Sh, SoT%, Save%, etc.) are **not** divided by 90s.

In [ ]:
for name, df in raw_dfs.items():
    df['90s'] = df['Min'] / 90

PER90_COLS = ['Gls', 'Ast', 'Sh', 'SoT', 'TklW', 'Int', 'Fls', 'Fld', 'CrdY', 'Off', 'PKwon', 'PKcon', 'OG', 'CrdR']
PER90_GOALKEEPERS = ['GA', 'SoTA', 'Saves', 'CS']
for name, df in raw_dfs.items():
    cols = PER90_COLS + (PER90_GOALKEEPERS if name == 'Goalkeepers' else [])
    for col in cols:
        if col in df.columns:
            df[f'{col}_p90'] = df[col] / df['90s']
    df = df.drop(columns=[c for c in cols if c in df.columns])
    raw_dfs[name] = df
    new_p90 = [f'{c}_p90' for c in cols if f'{c}_p90' in df.columns]
    print(f'{name}: built {len(new_p90)} per-90 features')

## Step E — Structural zeros in rate stats

G/Sh, G/SoT, SoT%, Save%_PK are NaN when denominator is 0. Add binary indicator and fill rate with 0.

In [ ]:
STRUCTURAL_RATE_COLS = {
    'G/Sh': ['Defenders', 'Midfielders_Forwards'],
    'G/SoT': ['Defenders', 'Midfielders_Forwards'],
    'SoT%': ['Defenders', 'Midfielders_Forwards'],
    'Save%_PK': ['Goalkeepers'],
}
for col, applies_to in STRUCTURAL_RATE_COLS.items():
    for name in applies_to:
        if name not in raw_dfs:
            continue
        df = raw_dfs[name]
        if col not in df.columns:
            continue
        indicator_name = col.replace('/', '_').replace('%', 'Pct') + '_zero'
        df[indicator_name] = df[col].isna().astype(int)
        null_count = df[col].isna().sum()
        df[col] = df[col].fillna(0)
        raw_dfs[name] = df
        print(f'{name} | {col}: {null_count} structural zeros flagged and filled')

## Step F — Dependent variable and Age²

log_wage = log(Annual_Wages_EUR). Age² for career arc. Drop Weekly_Wages_EUR (redundant with Annual).

In [ ]:
for name, df in raw_dfs.items():
    df['log_wage'] = np.log(pd.to_numeric(df['Annual_Wages_EUR'], errors='coerce'))
    valid = df['log_wage'].dropna()
    print(f'{name}: log_wage mean={valid.mean():.3f} std={valid.std():.3f}')
    print(f'  Implied annual wage at mean: EUR {np.exp(valid.mean()):,.0f}')
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df['Age2'] = df['Age'] ** 2
    if 'Weekly_Wages_EUR' in df.columns:
        df = df.drop(columns=['Weekly_Wages_EUR'])
    raw_dfs[name] = df

## Step G — Midfielders_Forwards position split (methodology)

**1. Unit of classification: player–season**  
Each row is one player in one season. We assign a position group **per row**, not per player. So the same player can be **Mediocampista** in one season and **Delantero** in another if FBRef changes their primary role (e.g. MF,FW one year, FW the next). This is correct for wage regression: we want positional homogeneity within each observation, not a fixed label per name.

**2. Primary position = first token of Pos — is the order meaningful?**  
FBRef stores positions as comma-separated (e.g. `MF,FW`, `FW,MF`, `DF,MF`). The **first token** is the player’s **primary (dominant) role** for that season. We define `Pos_primary` as that first token. (Empirical check: FW,MF have ~36% higher mean goals/90 and ~19% higher mean wages than MF,FW, so the order is informative.)

**3. Why DF,MF → Midfielders**  
The **Midfielders and Forwards** file only contains players whose roles are midfield and/or forward (pure centre-backs are in the Defenders file). So when we see **DF** in this file, it denotes a **defensive midfielder** (e.g. holding mid), not a central defender. We assign primary **DF** to **Midfielders** so that the midfield wage model includes all central, non-forward roles (MF and defensive MF). The Forwards model is restricted to primary **FW** so that we have a homogeneous “forward” production function.

**4. Assignment rule**  
- **Midfielders:** `Pos_primary` in {MF, DF}  
- **Forwards:** `Pos_primary` = FW  
No other values appear in this file after cleaning. Below we print the `Pos_primary` breakdown and a check that the same player can appear in different groups across seasons.

In [ ]:
# Classification is per player-season: same player can be MF one year, FW another (see methodology above).
df_mf_fw = raw_dfs['Midfielders_Forwards'].copy()
df_med['Pos_primary'] = df_med['Pos'].astype(str).str.split(',').str[0].str.strip()
# Continuous forwardness (0=no forward role, 1=pure forward) for role-weighted regression / interactions
FORWARDNESS_MAP = {'FW': 1.0, 'FW,MF': 0.7, 'MF,FW': 0.3, 'MF': 0.0, 'MF,DF': 0.0, 'DF,MF': 0.0, 'DF,FW': 0.2, 'FW,DF': 0.5}
pos_stripped = df_med['Pos'].astype(str).str.strip()
df_med['forwardness'] = pos_stripped.map(FORWARDNESS_MAP).fillna(pos_stripped.apply(lambda s: 1.0 if s == 'FW' else 0.0))
print('Pos_primary breakdown (per row):')
print(df_med['Pos_primary'].value_counts().to_string())
# Midfielders = primary MF or DF (defensive mid); Forwards = primary FW
df_mf = df_med[df_med['Pos_primary'].isin(['MF', 'DF'])].copy()
df_fw = df_med[df_med['Pos_primary'].isin(['FW'])].copy()
print(f'\nMidfielders (primary MF or DF): {len(df_mf)} rows')
print(f'Forwards (primary FW): {len(df_fw)} rows')
# Sanity: same player can have different Pos_primary across seasons
players_multi = df_med.groupby('Player')['Pos_primary'].nunique()
n_switch = (players_multi > 1).sum()
print(f'Players with different Pos_primary across seasons: {n_switch} (expected: role can change by season).')
print('\nForwardness (for role-weighted regression):')
print(df_med.groupby(pos_stripped)['forwardness'].first().to_string())
assert len(df_mf) + len(df_fw) == len(df_med), 'Rows lost in split!'
raw_dfs['Midfielders'] = df_mf
raw_dfs['Forwards'] = df_fw
del raw_dfs['Midfielders_Forwards']

### Role weights (alternative to a hard split)

A **hard split** (Midfielders vs Forwards) treats MF,FW and MF the same, so we ignore that MF,FW has a forward component. A more flexible approach is to give **different weights to variables by role**: e.g. goals matter more for forward-like players, defensive stats more for midfielder-like players.

We add a continuous **forwardness** score (0 = no forward role, 1 = pure forward) so that in regression you can use **interactions** (e.g. `Gls_p90 × forwardness`): the coefficient on goals then depends on role, without forcing each observation into one of two buckets. Mapping (transparent, based on primary + secondary role):

| Pos   | forwardness | Rationale |
|-------|-------------|-----------|
| FW    | 1.0 | Pure forward |
| FW,MF | 0.7 | Forward first, some mid |
| MF,FW | 0.3 | Mid first, some forward |
| MF    | 0.0 | Pure mid |
| MF,DF | 0.0 | Defensive mid |
| DF,MF | 0.0 | Defensive mid |
| DF,FW | 0.2 | Defender who also plays forward (rare) |
| FW,DF | 0.5 | Forward who also plays back (rare) |

Both Midfielders and Forwards model files will contain **forwardness**; use it in a single midfielders_forwards-wide regression with interactions to let variable importance vary by role.

## Step H — Final column audit per model

List identifiers (kept but not regressors) and regressors with null rate for methodology reference.

In [ ]:
IDENTIFIER_COLS = ['Player', 'Team', 'Season', 'Nation', 'Pos', 'Pos_primary', 'Annual_Wages_EUR', 'log_wage',
                   'MP', 'Min', '90s', 'Starts', 'Subs', 'unSub']
for name, df in raw_dfs.items():
    print(f'\n{"="*55}')
    print(f'MODEL: {name} ({len(df)} rows)')
    print(f'{"="*55}')
    identifiers = [c for c in IDENTIFIER_COLS if c in df.columns]
    regressors = [c for c in df.columns if c not in IDENTIFIER_COLS]
    print(f'Identifiers ({len(identifiers)}): {identifiers}')
    print('Regressors:')
    for c in regressors:
        null_rate = df[c].isna().mean()
        flag = ' *** NULL >5%' if null_rate > 0.05 else ''
        print(f'  {c:<30} null={null_rate:.1%}{flag}')

## Step I — Save model files

In [ ]:
MODEL_FILENAMES = {
    'Defenders': 'model_defenders.xlsx',
    'Midfielders': 'model_midfielders.xlsx',
    'Forwards': 'model_forwards.xlsx',
    'Goalkeepers': 'model_goalkeepers.xlsx',
}
for name, df in raw_dfs.items():
    out = MODELS_DIR / MODEL_FILENAMES[name]
    df.to_excel(out, index=False, engine='openpyxl')
    print(f'Saved {out.name} ({len(df)} rows x {df.shape[1]} cols)')
print('\nDone. Four model files written to data/processed/feature_engineered/')